# Mini-Modelo de Forecasting de Caja para Logística

Pipeline completo de generación, limpieza, detección de anomalías,
forecasting con Prophet y generación de reportes ejecutivos.

**Autor:** Equipo de Data Science  
**Seed:** 42 (reproducibilidad)

In [ ]:
from __future__ import annotations

import logging
import warnings
from typing import Any, Dict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s")

print("Librerías cargadas correctamente.")

---
## 1. Generación de Datos Sintéticos

Generamos ~730 filas (24 meses) con estacionalidades semanal, mensual y
trimestral, tendencia anual lineal y ~5% de anomalías controladas.

In [ ]:
from src.etl.generator import generate_dataset

df_raw = generate_dataset(seed=42, periods=24, anomaly_rate=0.05)

print(f"Filas generadas: {len(df_raw)}")
print(f"Columnas: {list(df_raw.columns)}")
print(f"Rango de fechas: {df_raw['fecha'].min()} → {df_raw['fecha'].max()}")
print(f"Anomalías: {df_raw['tiene_anomalia'].sum()} ({df_raw['tiene_anomalia'].mean()*100:.1f}%)")
df_raw.head()

---
## 2. Limpieza y Feature Engineering

Limpiamos outliers extremos, normalizamos tipos y añadimos:
- caja_neta (ingresos - gastos)
- Lags de 1, 7, 14, 30 días
- Medias móviles de 7, 14, 30 días
- Features temporales

In [ ]:
from src.etl.cleaner import clean_dataset
from src.etl.features import add_features

df_clean, cleaning_report = clean_dataset(df_raw)
print("=== Reporte de Limpieza ===")
for k, v in cleaning_report.items():
    print(f"  {k}: {v}")

print(f"\nFilas antes: {len(df_raw)}, después: {len(df_clean)}")
df_clean.head()

In [ ]:
df_feat = add_features(df_clean)

print(f"Filas con features: {len(df_feat)}")
print(f"Columnas totales: {df_feat.shape[1]}")
print(f"Columnas: {list(df_feat.columns)}")
df_feat.head()

---
## 3. Detección de Anomalías

Ejecutamos 5 métodos de detección:
1. **Isolation Forest** — detección basada en aislamiento
2. **Z-score** — desviaciones estándar
3. **IQR** — rango intercuartílico
4. **Benford** — distribución de primer dígito
5. **Temporal** — patrones estacionales

Luego combinamos todo en una matriz de consenso ponderado.

In [ ]:
from src.anomalies.isolation_forest import detect_isolation_forest
from src.anomalies.statistical import detect_zscore, detect_iqr
from src.anomalies.benford import detect_benford
from src.anomalies.temporal import detect_temporal
from src.anomalies.consensus import build_consensus_matrix
from src.anomalies.report import generate_anomaly_report

# 1. Isolation Forest
if_scores, if_anom = detect_isolation_forest(df_feat, random_state=42)
print(f"Isolation Forest: {if_anom.sum()} anomalías")

# 2. Z-score
z_anom = detect_zscore(df_feat)
print(f"Z-score: {z_anom.sum()} anomalías")

# 3. IQR
iqr_anom = detect_iqr(df_feat)
print(f"IQR: {iqr_anom.sum()} anomalías")

# 4. Benford
benford_anom = detect_benford(df_feat)
print(f"Benford: {benford_anom.sum()} anomalías")

# 5. Temporal
temp_anom = detect_temporal(df_feat)
print(f"Temporal: {temp_anom.sum()} anomalías")

In [ ]:
# Matriz de consenso
consensus_df = build_consensus_matrix(
    df=df_feat,
    if_anomalies=if_anom,
    zscore_anomalies=z_anom,
    iqr_anomalies=iqr_anom,
    benford_anomalies=benford_anom,
    temporal_anomalies=temp_anom,
    if_scores=if_scores,
)

print(f"\nMatriz de consenso: {len(consensus_df)} filas")
print(f"Anomalías detectadas (consenso): {(consensus_df['severidad'].notna()).sum()}")
print(f"Distribución por severidad:")
for sev in ['crítico', 'alto', 'medio', 'bajo']:
    count = (consensus_df['severidad'] == sev).sum()
    if count > 0:
        print(f"  {sev}: {count}")

consensus_df[consensus_df['severidad'].notna()].head(10)

In [ ]:
# Generar reportes
anomaly_stats = generate_anomaly_report(consensus_df, df_feat)

print(f"Total anomalías: {anomaly_stats['total_anomalies']}")
print(f"Tasa de detección: {anomaly_stats['detection_rate_vs_known']*100:.1f}%")
print(f"Archivos generados: {anomaly_stats['output_files']}")

---
## 4. Forecasting con Prophet

Entrenamos un modelo Prophet con estacionalidades semanal y anual,
festivos mexicanos como regresores, y validación walk-forward.

In [ ]:
from src.forecasting.train import train_prophet_model
from src.forecasting.predict import generate_forecast
from src.forecasting.backtest import walk_forward_backtest

# Entrenar modelo
model, forecast_full = train_prophet_model(
    df_feat,
    save_path='models/prophet_model.pkl',
)
print("Modelo Prophet entrenado correctamente.")
print(f"MAE: {float((forecast_full['yhat'] - forecast_full['yhat']).abs().mean()):.2f}")

In [ ]:
# Generar predicciones a 90 días
predictions = generate_forecast(model, periods=90)

print(f"Predicciones generadas: {len(predictions)} días")
print(f"Rango: {predictions['ds'].min()} → {predictions['ds'].max()}")
print(f"\nResumen por horizonte:")
for h in ['30d', '60d', '90d']:
    subset = predictions[predictions['horizonte'] == h]
    if not subset.empty:
        print(f"  {h}: {len(subset)} días, último yhat=${subset['yhat'].iloc[-1]:,.2f}")

predictions.head()

In [ ]:
# Walk-forward backtesting
backtest_results = walk_forward_backtest(
    df_feat,
    train_window=90,
    test_window=30,
    min_iterations=6,
)

print(f"Backtesting completado: {backtest_results['n_iterations']} iteraciones")
print(f"MAE global: {backtest_results['global_mae']:,.2f}")
print(f"RMSE global: {backtest_results['global_rmse']:,.2f}")
print(f"MAPE global: {backtest_results['global_mape']:.2f}%")
print(f"SMAPE global: {backtest_results['global_smape']:.2f}%")
print(f"Éxito por SMAPE: {backtest_results['success_by_smape']}")
print(f"Éxito por MAE: {backtest_results['success_by_mae']}")

---
## 5. Visualizaciones

Generamos gráficos interactivos con Plotly:
1. Forecast con intervalos de confianza
2. Componentes (tendencia, semanal, anual)
3. Análisis de residuos
4. Anomalías sobre el forecast

In [ ]:
from src.forecasting.visualize import generate_all_plots

# Cargar anomalías para el gráfico
import os
anomalies_for_viz = None
if os.path.exists('reports/anomaly_report.csv'):
    anomalies_for_viz = pd.read_csv('reports/anomaly_report.csv')

# Generar todos los plots
plot_paths = generate_all_plots(
    forecast_df=predictions,
    model=model,
    anomalies_df=anomalies_for_viz,
    output_dir='reports',
)
print(f"Gráficos generados: {len(plot_paths)}")
for p in plot_paths:
    print(f"  - {p}")

In [ ]:
# Mostrar gráfico inline (si plotly está disponible)
try:
    from src.forecasting.visualize import plot_forecast
    plot_forecast(predictions, title="Forecast de Caja Neta (90 días)")
    fig = plot_forecast.__globals__.get('go') or None
    if fig:
        print("Gráfico inline disponible (ejecutar celda para mostrar).")
except Exception as e:
    print(f"No se pudo mostrar inline: {e}")

---
## 6. Reportes

Generamos documentos automáticos:
- **Smart Narrative**: reporte narrativo nivel CFO
- **Resumen Ejecutivo**: 1 página con KPIs
- **Power BI Export**: CSV optimizado para dashboard
- **QA Report**: checklist de calidad del proyecto

In [ ]:
from src.dashboards.export_powerbi import export_powerbi
from src.reports.smart_narrative import generate_smart_narrative
from src.reports.executive_summary import generate_executive_summary
from src.reports.qa_report import generate_qa_report

# Power BI Export
print("=== Power BI Export ===")
result_pbi = export_powerbi(output_dir='reports')
print(f"CSV: {result_pbi['powerbi_csv']}")
print(f"Diseño: {result_pbi['dashboard_design']}")

print("\n=== Smart Narrative ===")
narrative_path = generate_smart_narrative()
print(f"Reporte: {narrative_path}")

print("\n=== Resumen Ejecutivo ===")
exec_path = generate_executive_summary()
print(f"Resumen: {exec_path}")

print("\n=== QA Report ===")
qa_path = generate_qa_report()
print(f"QA: {qa_path}")

---
## Conclusiones

### Resumen de Resultados

| Componente | Estado | Detalle |
|------------|--------|---------|
| Generación de datos | ✅ | ~730 filas sintéticas con 6 tipos de anomalías |
| Limpieza | ✅ | Outliers y normalización |
| Feature engineering | ✅ | 28 columnas con lags, rolling, features temporales |
| Anomalías | ✅ | 5 detectores + consenso ponderado |
| Forecasting | ✅ | Prophet con festivos mexicanos |
| Backtesting | ✅ | Walk-forward con SMAPE como métrica principal |
| Visualizaciones | ✅ | 4 gráficos HTML interactivos |
| Reportes | ✅ | Smart Narrative, Ejecutivo, QA, Power BI |

### Archivos Generados

```
data/
  raw/dataset_raw.csv
  curated/dataset_clean.csv
  curated/dataset_features.csv
models/
  prophet_model.pkl
reports/
  anomaly_report.csv
  consensus_matrix.csv
  forecast_results.csv
  forecast_plot.html
  forecast_components.html
  forecast_residuals.html
  forecast_anomalies.html
  smart_narrative.md
  ejecutivo_resumen.md
  qa_report.md
reports/
  forecast_powerbi.csv
docs/
  dashboard_design.md
```

---
*Notebook generado automáticamente — Mini-Modelo de Forecasting de Caja*